# Query Engines Explained

A query engine is the interface that turns a natural-language question into **retrieval** (finding relevant nodes) plus **synthesis** (turning those nodes into an answer). Both steps have settings worth knowing — this episode covers the two most impactful ones: `similarity_top_k` and `response_mode`.


First, the usual setup: quiet the logs, load environment variables, and set the default LLM/embedding model via `Settings`.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# LlamaIndex and its HTTP client (httpx) log a lot of INFO-level noise by default —
# bump both up to WARNING so only real problems show up in the output below.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Reads the .env file and copies its keys (e.g. OPENAI_API_KEY) into os.environ.
load_dotenv()

# Set the default LLM and embedding model every index/query engine below will use.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 1 — Build the baseline index.** Load the anime corpus and build a `VectorStoreIndex`, exactly as in earlier episodes — this is the index every query engine variation below will be built on top of.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load the anime corpus and build a VectorStoreIndex from it.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)
print(f"Index ready with {len(index.docstore.docs)} nodes")

Index ready with 8 nodes


**Step 2 — Tune `similarity_top_k`.** This setting controls how many chunks get retrieved per query. Running the same question at `top_k=2` vs. `top_k=5` shows how retrieving more context can pull in details spread across multiple documents.


In [3]:
top_k_question = "What signature techniques or transformations do these anime " "protagonists use?"

# similarity_top_k controls how many of the most similar nodes get retrieved and
# handed to the LLM — compare a small vs. larger value on the same question.
for top_k in (2, 5):
    engine = index.as_query_engine(similarity_top_k=top_k)
    response = engine.query(top_k_question)
    print(f"--- similarity_top_k={top_k} | {len(response.source_nodes)} source nodes ---")
    print(f"{len(str(response))}: {response}", end="\n\n")

--- similarity_top_k=2 | 2 source nodes ---
444: The protagonists use several signature techniques and transformations. One character employs Water Breathing and later awakens Sun Breathing, tied to his family's dance ritual. Another character uses Blood Demon Art techniques that produce explosive effects. In a different series, a protagonist is known for the Kamehameha energy beam and various Super Saiyan transformations, including Super Saiyan God, Super Saiyan Blue, and Ultra Instinct.

--- similarity_top_k=5 | 5 source nodes ---
625: The protagonists use a variety of signature techniques and transformations. Tanjiro Kamado employs Water Breathing and later awakens Hinokami Kagura, also known as Sun Breathing, which is tied to his family's dance ritual. Son Goku is known for the Kamehameha energy beam and his Super Saiyan transformations, including Super Saiyan God, Super Saiyan Blue, and Ultra Instinct. Naruto Uzumaki uses the Rasengan and Shadow Clone Jutsu, and later gains Sage M

**Step 3 — Tune `response_mode`.** This setting controls how retrieved chunks get synthesized into a final answer. Comparing `compact` against `tree_summarize` on a broad "summarize everything" question shows why the synthesis strategy matters, not just retrieval.


In [4]:
mode_question = "Summarize the story and central conflict of each of these anime series."

# response_mode controls HOW retrieved nodes get turned into an answer:
# "compact" stuffs as many nodes as fit into as few LLM calls as possible,
# "tree_summarize" recursively summarizes nodes in a tree, often better for broad questions.
for mode in ("compact", "tree_summarize"):
    engine = index.as_query_engine(similarity_top_k=5, response_mode=mode)
    response = engine.query(mode_question)
    print(f"--- response_mode={mode} ---")
    print(response, end="\n\n")

--- response_mode=compact ---
Demon Slayer follows Tanjiro Kamado, a kind-hearted boy who becomes a demon slayer after his family is slaughtered and his sister Nezuko is turned into a demon. The series explores his journey to find a cure for Nezuko and seek revenge, with key story arcs including the Final Selection, Mount Natagumo, Entertainment District, and Infinity Castle, culminating in a confrontation with the primary antagonist Muzan Kibutsuji.

Death Note centers on Light Yagami, a high school student who discovers a supernatural notebook that kills anyone whose name is written in it. The core conflict is a tense cat-and-mouse game between Light, who seeks to create a crime-free world under the alias "Kira," and the detective L, tasked with stopping him. The series delves into themes of justice, morality, and the corrupting influence of power.

Naruto follows Naruto Uzumaki, a young ninja from the Hidden Leaf Village who dreams of becoming Hokage. The story covers his growth, st

**Step 4 — Note the async option.** `.query()` is synchronous and blocks until the LLM responds; production code serving concurrent requests (e.g. a web API) should reach for the async equivalent instead.


In [ ]:
# .query() is synchronous and blocks until the LLM responds — fine for scripts
# and notebooks. In production (e.g. a FastAPI endpoint serving concurrent
# requests), use the async equivalent instead: `await query_engine.aquery(question)`.
print("Async equivalent available via query_engine.aquery() for production use.")

### Summary

- `similarity_top_k` controls how many chunks get retrieved — too low and the answer may miss relevant context spread across multiple documents; too high and you pay for tokens the LLM doesn't need.
- `response_mode` controls how retrieved chunks get synthesized into an answer — `compact` stuffs them into as few LLM calls as possible, while `tree_summarize` recursively summarizes in a tree, often better for broad "summarize everything" questions.
- These two settings are the first things worth tuning when a query engine's answers feel incomplete or unfocused.
